# 🎬 VideoClip Creator IA — Orquestador + Worker VIDEO UNIFICADO
### Todo en un solo Colab: análisis de audio, Gemini, Pollinations, LTX-Video, ensamblado

**PASO 1:** Runtime → Change runtime type → **GPU T4**
**PASO 2:** Ejecuta cada celda en orden (Ctrl+F9)
**PASO 3:** Copia la URL GRANDE que aparece al final y pégala en el Lanzador

In [ ]:
# ⚙️ CELDA 1/5: Instalar TODO (5-8 min)
!pip install -q flask flask-cors fastapi "uvicorn[standard]" nest-asyncio
!pip install -q librosa requests numpy soundfile
!pip install -q diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg pillow
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
print('✅ TODO instalado')

In [ ]:
# 🔐 CELDA 2/5: Configurar tokens (CAMBIA ESTOS VALORES)

# Token que el Lanzador usa para autenticarse (copia el del .env)
VCC_TOKEN = "5a9fba…0409"  # ← CAMBIA si rotaste

# Gemini API Key (GRATIS en https://aistudio.google.com/apikey)
GEMINI_API_KEY = "AIzaSy…cion"  # ← PEGA TU KEY REAL AQUÍ

GEMINI_MODEL = "gemini-2.5-flash"
WORKER_TOKEN = "ecb39caf280c95672f1dc730311c4ea526100689d7cacbdf"  # ← CAMBIA si rotaste

print('✅ Configuración cargada')
print(f'   VCC_TOKEN: {"OK" if VCC_TOKEN != "PENDIENTE" else "⚠️ PENDIENTE"}')
print(f'   Gemini Key: {"OK" if len(GEMINI_API_KEY) > 30 else "⚠️ PENDIENTE"}')
print(f'   Worker Token: {"OK" if WORKER_TOKEN != "PENDIENTE" else "⚠️ PENDIENTE"}')

In [ ]:
# 🤖 CELDA 3/5: Cargar LTX-Video (3-5 min)
import torch
from diffusers import LTXImageToVideoPipeline

pipe = LTXImageToVideoPipeline.from_pretrained(
    'Lightricks/LTX-Video',
    torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()
print('✅ LTX-Video listo')

In [ ]:
# 🚀 CELDA 4/5: Arrancar el servidor unificado
import base64, io, json, os, subprocess, threading, time, uuid, urllib.parse
from pathlib import Path

import requests
import nest_asyncio
import uvicorn
from fastapi import FastAPI, Form, Header, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse
from diffusers.utils import export_to_video
from PIL import Image

# ---- Configuración del servidor ----
BASE = Path("/content/proyectos")
BASE.mkdir(exist_ok=True)

TOKEN = VCC_TOKEN

app = FastAPI(title="VideoClip Creator IA — Unificado")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

# ---- Utilidades ----
def auth(x_token: str = Header(default="")):
    if x_token != TOKEN:
        raise HTTPException(401, "Token inválido")

def ruta_proy(pid: str) -> Path:
    p = BASE / pid
    if not p.exists():
        raise HTTPException(404, "Proyecto no existe")
    return p

def cargar(pid: str) -> dict:
    return json.loads((ruta_proy(pid) / "estado.json").read_text(encoding="utf-8"))

def guardar(pid: str, estado: dict):
    (ruta_proy(pid) / "estado.json").write_text(
        json.dumps(estado, indent=2, ensure_ascii=False), encoding="utf-8")

def gemini(prompt: str):
    if not GEMINI_API_KEY or len(GEMINI_API_KEY) < 30:
        raise HTTPException(500, "Falta GEMINI_API_KEY")
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent?key={GEMINI_API_KEY}"
    r = requests.post(url, json={
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"response_mime_type": "application/json"},
    }, timeout=180)
    r.raise_for_status()
    txt = r.json()["candidates"][0]["content"]["parts"][0]["text"]
    return json.loads(txt)

def imagen_gratis(prompt: str, destino: Path, w: int, h: int, seed: int = 42):
    url = ("https://image.pollinations.ai/prompt/"
           + urllib.parse.quote(prompt[:1400])
           + f"?width={w}&height={h}&nologo=true&seed={seed}&model=flux")
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    destino.write_bytes(r.content)

def ffmpeg(*args: str):
    cmd = ["ffmpeg", "-y", *args]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise HTTPException(500, "FFmpeg error: " + proc.stderr[-400:])

# ---- 1. Crear proyecto + análisis ----
@app.post("/api/proyecto")
def nuevo_proyecto(audio: UploadFile, letra: str = Form(""),
                   formato: str = Form("16:9"), x_token: str = Header(default="")):
    auth(x_token)
    pid = uuid.uuid4().hex[:12]
    carpeta = BASE / pid
    carpeta.mkdir()
    audio_path = carpeta / "audio" + (Path(audio.filename or ".mp3").suffix or ".mp3")
    audio_path.write_bytes(audio.file.read())

    bpm, duracion, beats = 0, 0.0, []
    try:
        import librosa
        y, sr = librosa.load(str(audio_path), sr=22050, mono=True)
        duracion = float(librosa.get_duration(y=y, sr=sr))
        tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
        bpm = round(float(tempo[0] if hasattr(tempo, "__len__") else tempo))
        beats = [round(float(t), 3) for t in librosa.frames_to_time(beat_frames, sr=sr)]
    except Exception:
        pass
    if duracion == 0:
        try:
            out = subprocess.run(["ffprobe", "-v", "quiet", "-show_entries", "format=duration",
                                  "-of", "csv=p=0", str(audio_path)],
                                 capture_output=True, text=True).stdout.strip()
            duracion = float(out)
        except Exception:
            duracion = 180.0

    estado = {"id": pid, "letra": letra, "formato": formato, "bpm": bpm,
              "duracion": round(duracion, 1), "beats": beats[:600],
              "audio": audio_path.name, "conceptos": [], "concepto": None,
              "personaje": None, "escenas": [], "final": None}
    guardar(pid, estado)
    return {"id": pid, "bpm": bpm, "duracion": estado["duracion"], "formato": formato}

# ---- 2. Conceptos ----
@app.post("/api/proyecto/{pid}/conceptos")
def conceptos(pid: str, x_token: str = Header(default="")):
    auth(x_token)
    est = cargar(pid)
    data = gemini(f"""Eres un director de videoclips premiado. Letra:
"""{est['letra'][:3500]}"""
Propón 3 conceptos visuales MUY distintos (fotorrealista, anime, cyberpunk, noir, onírico).
Devuelve JSON: {{"conceptos":[{{"nombre":"...","descripcion":"2 frases",
"paleta":"colores","personaje":"descripción del protagonista"}}]}}""")
    est["conceptos"] = data["conceptos"][:3]
    guardar(pid, est)
    return {"conceptos": est["conceptos"]}

# ---- 3. Personaje ----
@app.post("/api/proyecto/{pid}/personaje")
def personaje(pid: str, body: dict, x_token: str = Header(default="")):
    auth(x_token)
    est = cargar(pid)
    idx = body.get("concepto")
    estilo = est["conceptos"][idx]["nombre"] if isinstance(idx, int) and est["conceptos"] else "cinematográfico"
    desc = body.get("descripcion", "cantante protagonista")
    prompt = (f"character sheet, three views (front, profile, full body) of {desc}, "
              f"{estilo} style, consistent character, neutral background")
    w, h = (1216, 704) if est["formato"] == "16:9" else (704, 1216)
    destino = ruta_proy(pid) / "personaje.png"
    imagen_gratis(prompt, destino, w, h, seed=7)
    est["personaje"] = {"descripcion": desc, "archivo": "personaje.png"}
    guardar(pid, est)
    return {"url": f"/api/proyecto/{pid}/archivo/personaje.png"}

# ---- 4. Storyboard ----
@app.post("/api/proyecto/{pid}/storyboard")
def storyboard(pid: str, body: dict, x_token: str = Header(default="")):
    auth(x_token)
    est = cargar(pid)
    idx = body.get("concepto")
    if isinstance(idx, int):
        est["concepto"] = idx
    concepto = est["conceptos"][est["concepto"]] if est["conceptos"] and est["concepto"] is not None else {}
    n_escenas = max(4, round(est["duracion"] / 8))
    dur_escena = round(est["duracion"] / n_escenas, 1)
    data = gemini(f"""Cancion de {est['duracion']}s, {est['bpm']} BPM. Concepto: {json.dumps(concepto, ensure_ascii=False)}
Personaje: {(est.get('personaje') or {}).get('descripcion', '')}
Letra: """{est['letra'][:3500]}"""
Crea {n_escenas} escenas de {dur_escena}s. Marca lipsync=true SOLO cuando el personaje canta a camara.
Prompt visual detallado en ingles, mencionando al personaje y estilo "{concepto.get('nombre','')}".
JSON: {{"escenas":[{{"seccion":"...","prompt":"...","lipsync":false}}]}}""")
    escenas = []
    for i, e in enumerate(data["escenas"][:n_escenas]):
        escenas.append({"n": i, "seccion": e.get("seccion", ""), "prompt": e["prompt"],
                        "duracion": dur_escena, "lipsync": bool(e.get("lipsync")),
                        "imagen": None, "video": None})
    est["escenas"] = escenas
    guardar(pid, est)
    return {"escenas": escenas}

# ---- 5a. Imagen de referencia ----
@app.post("/api/proyecto/{pid}/escenas/{i}/imagen")
def escena_imagen(pid: str, i: int, body: dict, x_token: str = Header(default="")):
    auth(x_token)
    est = cargar(pid)
    esc = est["escenas"][i]
    personaje_desc = (est.get("personaje") or {}).get("descripcion", "")
    prompt = body.get("prompt") or esc["prompt"]
    full = f"{prompt}. Main character: {personaje_desc}. Same character, cinematic still frame."
    w, h = (1216, 704) if est["formato"] == "16:9" else (704, 1216)
    destino = ruta_proy(pid) / f"escena_{i}.png"
    imagen_gratis(full, destino, w, h, seed=100 + i)
    esc["prompt"] = prompt
    esc["imagen"] = destino.name
    guardar(pid, est)
    return {"url": f"/api/proyecto/{pid}/archivo/{destino.name}"}

# ---- 5b. Animar escena con LTX-Video (GPU local en este mismo Colab) ----
@app.post("/api/proyecto/{pid}/escenas/{i}/video")
def escena_video(pid: str, i: int, body: dict, x_token: str = Header(default="")):
    auth(x_token)
    est = cargar(pid)
    esc = est["escenas"][i]
    img_path = ruta_proy(pid) / (esc.get("imagen") or f"escena_{i}.png")
    if not img_path.exists():
        raise HTTPException(400, "Primero genera la imagen de la escena")

    segundos = min(float(body.get("duracion") or esc["duracion"]), 8.0)
    fps = 24
    frames = int(segundos * fps) // 8 * 8 + 1

    img = Image.open(img_path).convert("RGB")
    img.thumbnail((768, 768))

    prompt = body.get("prompt") or esc["prompt"]
    prompt = prompt + ", cinematic, smooth motion, high quality, 4K"

    out_frames = pipe(
        image=img,
        prompt=prompt,
        num_frames=frames,
        frame_rate=fps,
        num_inference_steps=25,
        guidance_scale=3.0
    ).frames[0]

    destino = ruta_proy(pid) / f"escena_{i}.mp4"
    export_to_video(out_frames, str(destino), fps=fps)
    esc["video"] = destino.name
    guardar(pid, est)
    return {"url": f"/api/proyecto/{pid}/archivo/{destino.name}"}

# ---- 6. Ensamblado final ----
@app.post("/api/proyecto/{pid}/ensamblar")
def ensamblar(pid: str, x_token: str = Header(default="")):
    auth(x_token)
    est = cargar(pid)
    carpeta = ruta_proy(pid)
    clips = [carpeta / e["video"] for e in est["escenas"] if e.get("video")]
    if not clips:
        raise HTTPException(400, "No hay escenas generadas")

    lista = carpeta / "lista.txt"
    lista.write_text("".join(f"file '{c.name}'\n" for c in clips))

    sin_audio = carpeta / "sin_audio.mp4"
    ffmpeg("-f", "concat", "-safe", "0", "-i", str(lista),
           "-c:v", "libx264", "-pix_fmt", "yuv420p", "-an", str(sin_audio))

    final = carpeta / "videoclip_final.mp4"
    ffmpeg("-i", str(sin_audio), "-i", str(carpeta / est["audio"]),
           "-c:v", "copy", "-c:a", "aac", "-shortest", str(final))

    est["final"] = final.name
    guardar(pid, est)
    return {"url": f"/api/proyecto/{pid}/archivo/{final.name}"}

# ---- Consultas ----
@app.get("/api/proyecto/{pid}")
def estado(pid: str, x_token: str = Header(default="")):
    auth(x_token)
    return cargar(pid)

@app.get("/api/proyecto/{pid}/archivo/{nombre}")
def archivo(pid: str, nombre: str, x_token: str = Header(default="")):
    auth(x_token)
    if "/" in nombre or ".." in nombre:
        raise HTTPException(400, "Nombre inválido")
    p = ruta_proy(pid) / nombre
    if not p.exists():
        raise HTTPException(404, "Archivo no existe")
    return FileResponse(str(p))

# ---- Health check ----
@app.get("/api/health")
def health():
    return {"status": "ok", "workers": {"video": True, "lipsync": False}}

# ---- Arrancar ----
nest_asyncio.apply()
uv_thread = threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8080),
    daemon=True
)
uv_thread.start()
print('⏳ Servidor arrancando en puerto 8080...')
time.sleep(8)
print('✅ Servidor unificado corriendo')

In [ ]:
# 🌐 CELDA 5/5: Túnel público + URL GRANDE PARA COPIAR
import subprocess, re, time

print('⏳ Creando túnel público con Cloudflare...')
proc = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8080'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

url = None
for _ in range(120):
    line = proc.stdout.readline()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break

if not url:
    print('❌ ERROR: No se pudo crear el túnel')
else:
    print()
    print('=' * 60)
    print()
    print('    🎬 ¡COLAB LISTO!   Copia esta URL 👇')
    print()
    print(f'    🔗  {url}')
    print()
    print('    PÉGALA en el campo "URL del Orquestador"')
    print('    de la página Lanzador y guarda.')
    print()
    print('    NOTA: No cierres esta pestaña mientras generas.')
    print()
    print('=' * 60)
    print()
    
    # Mantener vivo
    while True:
        time.sleep(60)